# UK House Prices per Square Metre — Analysis

Replicates and updates Anna Powell-Smith's [houseprices.anna.ps](http://houseprices.anna.ps/) analysis using:
- HM Land Registry Price Paid Data through January 2026
- Energy Performance Certificates (current bulk download)
- UBDC PPD→UPRN lookup for UPRN-first joining

**Run the pipeline first**: `uv run python src/houseprices/pipeline.py`  
This notebook reads from `cache/matched.parquet` and `output/`.

In [ ]:
import json
import pathlib

import matplotlib.pyplot as plt
import pandas as pd

ROOT = pathlib.Path("..")  # notebook runs from notebooks/
CACHE = ROOT / "cache"
OUTPUT = ROOT / "output"
DATA = ROOT / "data"

# Verify pipeline outputs exist
for p in [
    CACHE / "matched.parquet",
    OUTPUT / "price_per_sqm_postcode_district.csv",
    OUTPUT / "price_per_sqm_lsoa.csv",
]:
    if not p.exists():
        raise FileNotFoundError(
            f"{p} not found. Run the pipeline first:\n"
            "  uv run python src/houseprices/pipeline.py"
        )

print("Pipeline outputs found ✓")

## 1. Load pipeline outputs

In [ ]:
matched = pd.read_parquet(
    CACHE / "matched.parquet",
    columns=["match_tier"],
)
districts = pd.read_csv(OUTPUT / "price_per_sqm_postcode_district.csv")
lsoa = pd.read_csv(OUTPUT / "price_per_sqm_lsoa.csv")

print(f"Matched records:       {len(matched):>10,}")
print(f"Postcode districts:    {len(districts):>10,}")
print(f"LSOAs:                 {len(lsoa):>10,}")

## 2. Match rate summary

In [ ]:
tier_counts = matched["match_tier"].value_counts().sort_index()
total = len(matched)

print("Match tier breakdown")
print("-" * 40)
for tier, count in tier_counts.items():
    pct = 100 * count / total
    label = "UPRN direct" if tier == 1 else "Address normalisation"
    print(f"  Tier {tier} ({label}): {count:>8,}  ({pct:.1f}%)")
print(f"  Total matched:          {total:>8,}")

lsoa_coverage = lsoa["num_sales"].sum()
print(f"\n  LSOA21CD assigned:      {lsoa_coverage:>8,}  ({100*lsoa_coverage/total:.1f}%)")

## 3. National overview

In [ ]:
median_ppsqm = districts["price_per_sqm"].median()
mean_ppsqm = districts["price_per_sqm"].mean()

top5 = districts.nlargest(5, "price_per_sqm")[["postcode_district", "price_per_sqm", "num_sales"]]
bottom5 = districts.nsmallest(5, "price_per_sqm")[["postcode_district", "price_per_sqm", "num_sales"]]

print(f"National median price/m²:  £{median_ppsqm:>6,.0f}")
print(f"National mean price/m²:    £{mean_ppsqm:>6,.0f}")
print(f"Districts covered:         {len(districts):>6,}")
print(f"Total sales:               {districts['num_sales'].sum():>6,}")

print("\n--- Most expensive districts ---")
display(top5.reset_index(drop=True))

print("--- Least expensive districts ---")
display(bottom5.reset_index(drop=True))

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
ax.hist(districts["price_per_sqm"], bins=80, edgecolor="none", color="steelblue")
ax.axvline(median_ppsqm, color="tomato", linewidth=1.5, label=f"Median £{median_ppsqm:,.0f}")
ax.set_xlabel("Price per m² (£)")
ax.set_ylabel("Number of postcode districts")
ax.set_title("Distribution of price per m² by postcode district")
ax.legend()
plt.tight_layout()
plt.savefig(OUTPUT / "price_per_sqm_distribution.png", dpi=150)
plt.show()

## 4. Comparison vs Anna Powell-Smith's analysis

Requires `data/anna_reference.json` — copy `data/anna_reference.json.example` and fill in her published figures.

In [ ]:
anna_path = DATA / "anna_reference.json"

if not anna_path.exists():
    print(
        "Skipping comparison: data/anna_reference.json not found.\n"
        "Run: uv run scripts/fetch_anna_reference.py"
    )
    anna = None
else:
    with anna_path.open() as f:
        anna = json.load(f)
    print(f"Anna's reference data loaded (fetched: {anna.get('fetched_date', 'unknown')})")

In [ ]:
if anna and anna.get("postcode_districts"):
    anna_districts = pd.DataFrame(
        [(k, v) for k, v in anna["postcode_districts"].items() if v is not None],
        columns=["postcode_district", "anna_price_per_sqm"],
    )

    comparison = districts.merge(anna_districts, on="postcode_district", how="inner")
    comparison["change_abs"] = comparison["price_per_sqm"] - comparison["anna_price_per_sqm"]
    comparison["change_pct"] = (
        100 * comparison["change_abs"] / comparison["anna_price_per_sqm"]
    ).round(1)

    matched_count = len(comparison)
    print(f"Matched {matched_count} districts to Anna's reference data")

    if anna.get("national_median_price_per_sqm"):
        our_median = districts["price_per_sqm"].median()
        anna_median = anna["national_median_price_per_sqm"]
        print(f"\nNational median: £{our_median:,.0f} (Anna: £{anna_median:,}, "
              f"{100*(our_median-anna_median)/anna_median:+.1f}%)")
else:
    print("No anna_reference data available — skipping comparison.")
    comparison = None

## 5. Biggest movers since Anna's analysis

In [ ]:
if comparison is not None:
    print("--- Biggest increases (£/m²) ---")
    display(
        comparison.nlargest(10, "change_abs")[
            ["postcode_district", "anna_price_per_sqm", "price_per_sqm", "change_abs", "change_pct"]
        ].reset_index(drop=True)
    )

    print("--- Biggest decreases (£/m²) ---")
    display(
        comparison.nsmallest(10, "change_abs")[
            ["postcode_district", "anna_price_per_sqm", "price_per_sqm", "change_abs", "change_pct"]
        ].reset_index(drop=True)
    )
else:
    print("No comparison data — populate data/anna_reference.json to see movers.")

## 6. LSOA analysis

Covers only the UPRN-matched subset (Tier 1). Not directly comparable to the postcode district output which covers all matched records.

In [ ]:
if len(lsoa) == 0:
    print("LSOA output is empty — no UPRN-matched records with LSOA assignments.")
    print("This is expected until the OS Open UPRN and boundary files are in data/.")
else:
    lsoa_median = lsoa["price_per_sqm"].median()
    print(f"LSOAs covered:          {len(lsoa):>8,}")
    print(f"Total LSOA sales:       {lsoa['num_sales'].sum():>8,}")
    print(f"Median price/m² (LSOA): £{lsoa_median:>6,.0f}")

    print("\n--- Most expensive LSOAs ---")
    display(
        lsoa.nlargest(10, "price_per_sqm")[
            ["LSOA21CD", "price_per_sqm", "num_sales"]
        ].reset_index(drop=True)
    )

## 7. Export comparison summary

In [ ]:
lines = [
    "# UK House Prices per Square Metre — Comparison Summary",
    "",
    f"PPD data through January 2026 joined to EPC using UBDC UPRN lookup + address normalisation.",
    "",
    "## Match rate",
    "",
]

for tier, count in tier_counts.items():
    label = "UPRN direct (Tier 1)" if tier == 1 else "Address normalisation (Tier 2)"
    lines.append(f"- {label}: {count:,} ({100*count/total:.1f}%)")

lines += [
    "",
    "## National overview",
    "",
    f"- Median price per m²: £{median_ppsqm:,.0f}",
    f"- Postcode districts covered: {len(districts):,}",
    f"- Total sales: {districts['num_sales'].sum():,}",
    "",
]

if comparison is not None and anna.get("national_median_price_per_sqm"):
    anna_median = anna["national_median_price_per_sqm"]
    our_median = districts["price_per_sqm"].median()
    change_pct = 100 * (our_median - anna_median) / anna_median
    lines += [
        "## Comparison vs Anna Powell-Smith",
        "",
        f"- Anna's national median (fetched: {anna.get('fetched_date', 'unknown')}): £{anna_median:,}",
        f"- Current national median: £{our_median:,.0f} ({change_pct:+.1f}%)",
        "",
        "### Biggest increases (top 10)",
        "",
        comparison.nlargest(10, "change_abs")[
            ["postcode_district", "anna_price_per_sqm", "price_per_sqm", "change_pct"]
        ].to_markdown(index=False),
        "",
        "### Biggest decreases (top 10)",
        "",
        comparison.nsmallest(10, "change_abs")[
            ["postcode_district", "anna_price_per_sqm", "price_per_sqm", "change_pct"]
        ].to_markdown(index=False),
    ]

comparison_md = OUTPUT / "comparison.md"
comparison_md.write_text("\n".join(lines))
print(f"Written → {comparison_md}")